In [27]:
import pymupdf
import re


def is_toc_page_fast(page):
    text = page.get_text("text")

    if not text:
        return False

    text_lower = text.lower()

    if "table of contents" in text_lower or text_lower.strip().startswith("contents"):
        return True

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    pattern = re.compile(r".+\.{2,}\s*\d+$|.+\s+\d+$")
    matches = sum(1 for line in lines if pattern.match(line))

    if len(lines) > 3 and matches / len(lines) > 0.2:
        return True

    return False


def find_toc_pages_pymupdf(pdf_path):
    toc_pages = []

    doc = pymupdf.open(pdf_path)

    for i in range(len(doc)):
        page = doc[i]

        if is_toc_page_fast(page):
            toc_pages.append(i)
        elif toc_pages:
            break  #The assumption is that the table of contents is contiguous
    doc.close()

    return toc_pages

In [ ]:
toc_pages = find_toc_pages_pymupdf("example1.pdf")

print(toc_pages)

[5, 6, 7, 8]


In [ ]:
system_prompt_toc_parser = """
You are a meticulous document boundary analyst. You will receive:
- A list of texts. Each item in the list represents a page of the table of contents in a book.

Your task:
- Read through the contents of the page. 
- Determine the most likely start page where the subchapter begins (usually at the first occurrence of its heading/title or its numbering)
- Determine the most likely end page where the subchapter ends. The end page is typically the page immediately before the next subchapter's start page. If the subchapter is not present in the window, end at the last page where the subchapter's content is predominant.
- Use textual evidence and be resilient to OCR errors.
- If evidence is ambiguous, use logic and the page number of adjacent chapters to provide best effort estimates

Output requirements (CRITICAL):
- Output the data into valid JSON (no markdown, no extra keys beyond provided schema).
- Include the confidence as a number from 0 to 1
- Never invent pages outside the provided page range. Make sure that page numbers at most overlap by only 1 page (to account for a chapter ending and a new chapter starting in the same page)

JSON schema:
[
    {
        'chapter_name': 'Chapter 2. Managing Users and Permissions'
        'subchapter': 'Understanding the purpose of users and groups'
        'page_start': '32'
        'page_end': '32'
        'confidence': 0.923
    },
    {
        'chapter_name': 'Chapter 3. Navigating and Essential Commands'
        'subchapter': 'Learning the essential Linux commands'
        'page_start': '85'
        'page_end': '89'
        'confidence': 0.988
    },
]
Now produce the JSON for all provided TOC subchapter items, preserving the given order
"""

system_prompt_concepts_summarizer = """
You are an expert technical educator and summarizer. You will receive:
- A subchapter title (and optional numbering)
- The extracted text for that subchapter, broken into pages (page number + text)
- Possibly noisy OCR; preserve the meaning, not the exact wording

Your job:
- Produce a concise but faithful summary of the concepts introduced in the subchapter.
- Do NOT summarize everything verbatim; instead extract the key concepts, definitions, and relationships.
- Focus on “pre-read” value: what a learner needs to know before deeper reading.
- Use the page text as the source of truth. If something is unclear/contradicted, mention that uncertainty briefly.

Output requirements (CRITICAL):
- Output ONLY valid JSON (no markdown).
- Include a single concise paragraph that summarizes the key concepts introduced in the subchapter and explains their relationships at a learner-friendly level. Prioritize definitions, major claims, and the “so what / how it connects” ideas. Do not quote large passages verbatim; instead paraphrase. If the text is incomplete or ambiguous due to OCR, still provide the best-faith summary, but clearly and briefly note uncertainty in a natural way.

JSON Schema:
{
    'summary': 'We learn about simple Linux filesystem commands to help us become more confident in navigating the filesystem of our Ubuntu server. In this subchapter, the commands `cd`, `pwd`, `touch`, `rm`, `ls` and `ls -l` are introduced. 
}
"""

system_prompt_question_generator = """
You are a question-writing tutor for self-study. You will receive:
- A subchapter title
- Extracted page text, provided as a list of pages (each with a page number and text)
- A summary of the subchapter generated by a previous LLM

Your task:
- Generate approximately 5 questions per page that help the user pre-read and test understanding.
- Questions must be answerable from the provided page text (or based on explicit explanations present there).
- Mix question types to improve learning:
    - factual recall
    - conceptual explanation (“why/how”)
    - application/mini-scenarios
    - comparison/contrast (when the page supports it)
    - “common misconception” style (only if the text indicates it)
- Do NOT provide answers. Do NOT provide page-level summaries as answers.
- Keep questions clear even if OCR is imperfect; infer only when strongly supported.

Output requirements (CRITICAL):
- Output ONLY valid JSON (no markdown).
- For each question, include:
    - question (string)
    - type (one of: "recall" | "concept" | "application" | "comparison" | "misconception")
    - difficulty (1–3)
Use this schema:
{
    "title": "string",
    "questions": [
        {
            "question": "What command would you use to create a file in Linux?",
            "type": "recall, application",
            "difficulty": 1,
        },
        {
            "question": "What is the difference between `ls` and `ls -a`",
            "type": "comparison",
            "difficulty": 2,
        },
        {
            "question": "How would you update the 'last updated' time of a file without making any changes to it?",
            "type": "application",
            "difficulty": 3,
        }
    ]
}
"""

In [32]:
toc_pages

[5, 6, 7, 8]

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

class ChapterPages(BaseModel):
    chapter_name: Optional[str] = Field(
        default=None,
        description="Full chapter title including numbering (if present)"
    )
    subchapter: Optional[str] = Field(
        default=None,
        description="Subchapter or section title (if present)"
    )
    page_start: int = Field(
        description="Starting page number of the section"
    )
    page_end: int = Field(
        description="Ending page number of the section"
    )
    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence score between 0 and 1"
    )


class ChapterSummary(BaseModel):
    summary: str = Field(
        description="Summary of the chapter, condensed into one paragraph."
    )

class ChapterQuestion(BaseModel):
    question: str = Field(description="The question text")
    type: List[Literal["recall", "concept", "application", "comparison", "misconception"]] = Field(
        description="Type of question"
    )
    difficulty: int = Field(
        ge=1,
        le=5,
        description="Difficulty level from 1 (easy) to 5 (extremely hard)"
    )